## Basic Imports for Testing

In [6]:
import os
import finnhub
import pandas as pd
import json
import duckdb
from functools import reduce
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())
api_key = os.getenv('FINNHUB_API_KEY')

# Setup client
finnhub_client = finnhub.Client(api_key=api_key)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

def normalize_json_dataframe(df):
    """
    Iterates through each row in the DataFrame, normalizes JSON content in each column,
    and creates new DataFrames from the normalized JSON, associating each resulting row
    with the original index value.
    
    Assumes each cell contains a JSON string, dict, or list of dicts.
    Returns a list of DataFrames, one per original row.
    """
    result_dataframes = {}
    
    for idx, row in df.iterrows():
        row_normalized_dfs = []
        
        for col in df.columns:
            cell_value = row[col]
            
            # Parse JSON if it's a string
            if isinstance(cell_value, str):
                try:
                    json_data = json.loads(cell_value)
                except json.JSONDecodeError:
                    continue  # Skip invalid JSON
            else:
                json_data = cell_value
            
            # Normalize if it's a dict or list of dicts
            if isinstance(json_data, (dict, list)):
                try:
                    normalized_df = pd.json_normalize(json_data)
                    normalized_df['original_index'] = idx  # Add original index to the normalized DataFrame
                    rename_cols = [i for i in normalized_df.columns if i == 'v']
                    for i in rename_cols:
                        normalized_df.rename(columns={i: f"{col}_{i}"}, inplace=True)
                    row_normalized_dfs.append(normalized_df)
                except Exception:
                    continue  # Skip if normalization fails
        
        # Combine all normalized DataFrames for this row
        if row_normalized_dfs:
            combined_df = reduce(lambda left, right: pd.merge(left, right, on=['period', 'original_index'], how='outer'), row_normalized_dfs)
            ## combined_df.set_index('original_index', inplace=True)
            result_dataframes[idx] = combined_df
    
    return result_dataframes

## Free Plan Endpoints

##### Company Basic Financials

In [7]:
# Basic financials
## print(finnhub_client.company_basic_financials('AAPL', 'all'))
df = pd.DataFrame(finnhub_client.company_basic_financials('AAPL', 'all'))
annual_and_quarterly_pre_df = df.copy(deep=True)
annual_pre_df = pd.DataFrame(annual_and_quarterly_pre_df.loc['annual']).iloc[-2:,:].T.set_index('symbol') ## .reset_index()
annual_df = pd.json_normalize(annual_pre_df['series'])
quarterly_pre_df = pd.DataFrame(annual_and_quarterly_pre_df.loc['quarterly']).iloc[-2:,:].T.set_index('symbol') ## .reset_index()
quarterly_df = pd.json_normalize(quarterly_pre_df['series'])
df = df.drop(['annual','quarterly'], axis=0)
df = df.drop(columns=['metricType','series']).reset_index()

In [8]:
duckdb.register("df_view", df)
res = duckdb.query("SELECT * FROM df_view WHERE index == '10DayAverageTradingVolume'").to_df()
res

,index,metric,symbol
0,10DayAverageTradingVolume,41.52879,AAPL


In [9]:
symbol_name = 'AAPL'
annual_json_parsed_dfs = normalize_json_dataframe(annual_df)
duckdb.register("annual_json_parsed_df_view", annual_json_parsed_dfs[symbol_name])
res = duckdb.query("SELECT * FROM annual_json_parsed_df_view WHERE period == '1985-09-27'").to_df()
res


,period,bookValue_v,original_index,cashRatio_v,currentRatio_v,ebitPerShare_v,eps_v,ev_v,evEbitda_v,evRevenue_v,fcfMargin_v,grossMargin_v,inventoryTurnover_v,longtermDebtTotalAsset_v,longtermDebtTotalCapital_v,longtermDebtTotalEquity_v,netDebtToTotalCapital_v,netDebtToTotalEquity_v,netMargin_v,operatingMargin_v,payoutRatio_v,pb_v,pe_v,pfcf_v,pretaxMargin_v,ps_v,ptbv_v,quickRatio_v,receivablesTurnover_v,roa_v,roe_v,roic_v,rotc_v,salesPerShare_v,sgaToSale_v,tangibleBookValue_v,totalDebtToEquity_v,totalDebtToTotalAsset_v,totalDebtToTotalCapital_v,totalRatio_v
0,1985-09-27,NaN,AAPL,1.140773,2.7827,0.0074,0.0044,NaN,NaN,NaN,0.1099,0.4173,NaN,0.0,0.0,0.0,-0.6122,-0.6122,0.0319,0.0536,NaN,NaN,NaN,NaN,0.0626,NaN,NaN,2.2175,NaN,0.0654,0.1112,0.1112,0.1867,0.1384,0.5827,NaN,0.0,0.0,0.0,2.4273


In [10]:
symbol_name = 'AAPL'
quarterly_json_parsed_dfs = normalize_json_dataframe(quarterly_df)
duckdb.register("quarterly_json_parsed_df_view", quarterly_json_parsed_dfs[symbol_name])
res = duckdb.query("SELECT * FROM quarterly_json_parsed_df_view WHERE period == '1989-06-30'").to_df()
res

,period,assetTurnoverTTM_v,original_index,bookValue_v,cashRatio_v,currentRatio_v,ebitPerShare_v,eps_v,ev_v,evEbitdaTTM_v,evRevenueTTM_v,fcfMargin_v,fcfPerShareTTM_v,grossMargin_v,inventoryTurnoverTTM_v,longtermDebtTotalAsset_v,longtermDebtTotalCapital_v,longtermDebtTotalEquity_v,netDebtToTotalCapital_v,netDebtToTotalEquity_v,netMargin_v,operatingMargin_v,payoutRatioTTM_v,pb_v,peTTM_v,pfcfTTM_v,pretaxMargin_v,psTTM_v,ptbv_v,quickRatio_v,receivablesTurnoverTTM_v,roaTTM_v,roeTTM_v,roicTTM_v,rotcTTM_v,salesPerShare_v,sgaToSale_v,tangibleBookValue_v,totalDebtToEquity_v,totalDebtToTotalAsset_v,totalDebtToTotalCapital_v,totalRatio_v
0,1989-06-30,NaN,AAPL,NaN,NaN,NaN,0.0104,0.0066,NaN,NaN,NaN,0.1998,NaN,0.4941,NaN,NaN,NaN,NaN,NaN,NaN,0.077,0.1201,NaN,NaN,NaN,NaN,0.1263,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0862,0.5059,NaN,NaN,NaN,NaN,NaN


##### Earnings Surprises

In [19]:
# Earnings surprises
## print(finnhub_client.company_earnings('AAPL', limit=5))
df = pd.DataFrame(finnhub_client.company_earnings('AAPL'))
df

,actual,estimate,period,quarter,surprise,surprisePercent,symbol,year
0,2.84,2.7257,2025-12-31,1,0.1143,4.1934,AAPL,2026
1,1.85,1.8075,2025-09-30,4,0.0425,2.3513,AAPL,2025
2,1.57,1.4626,2025-06-30,3,0.1074,7.3431,AAPL,2025
3,1.65,1.6596,2025-03-31,2,-0.0096,-0.5785,AAPL,2025


##### Company News

In [37]:
# Company News
# Need to use _from instead of from to avoid conflict
df = pd.json_normalize(finnhub_client.company_news('AAPL', _from=pd.to_datetime("today").date(), to=pd.to_datetime("today").date()))
df['datetime_converted'] = pd.to_datetime(df['datetime'], unit='s')
df.drop(columns=['image'], inplace=True)
df

,category,datetime,headline,id,related,source,summary,url,datetime_converted
0,company,1774553280,Apple Stock Has Been Weak This Year. Why Analy...,139546888,AAPL,Yahoo,"Apple stock has been struggling this year, in...",https://finnhub.io/api/news?id=f7ec1176b941346...,2026-03-26 19:28:00
1,company,1774552560,"Apple iPhone loyalty strengthens, supporting s...",139546889,AAPL,Yahoo,"Apple Inc (NASDAQ:AAPL, XETRA:APC) iPhone ecos...",https://finnhub.io/api/news?id=c3e2a6b87990cf5...,2026-03-26 19:16:00
2,company,1774549830,"Apple plans to open Siri to rival AI services,...",139546890,AAPL,Yahoo,March 26 (Reuters) - Apple plans to open its S...,https://finnhub.io/api/news?id=05c31c9af364423...,2026-03-26 18:30:30
3,company,1774549487,Apple Plans to Open Up Siri to Rival AI Assist...,139546891,AAPL,Yahoo,(Bloomberg) -- Apple Inc. plans to open Siri t...,https://finnhub.io/api/news?id=a15938b49cce8b0...,2026-03-26 18:24:47
4,company,1774548482,Apple Taps TSMC Washington Plant As Valuation ...,139546892,AAPL,Yahoo,Apple is partnering with Taiwan Semiconductor ...,https://finnhub.io/api/news?id=43370059c1c420d...,2026-03-26 18:08:02
5,company,1774547280,Apple's Expanding Enterprise Footprint to Boos...,139546893,AAPL,Yahoo,AAPL's enterprise push accelerates as major cl...,https://finnhub.io/api/news?id=d9a8a531cea63e7...,2026-03-26 17:48:00
6,company,1774546431,Sector Update: Tech Stocks Fall Thursday After...,139546135,AAPL,Yahoo,"Tech stocks were lower Thursday afternoon, wit...",https://finnhub.io/api/news?id=e97fbb50aaa14f8...,2026-03-26 17:33:51
7,company,1774544998,Cirrus Logic Stock Breaks Out On Apple Partner...,139546136,AAPL,Yahoo,Cirrus Logic stock jumped after customer Apple...,https://finnhub.io/api/news?id=0e52505c5b6f04c...,2026-03-26 17:09:58
8,company,1774544160,Golden Eagle Strategies Releases first Hypergr...,139546137,AAPL,Yahoo,"Golden Eagle Strategies, LLC today announced t...",https://finnhub.io/api/news?id=e9613318a83829d...,2026-03-26 16:56:00
9,company,1774544098,"Identical Tech Exposure, Lower Cost or Greater...",139546120,AAPL,Yahoo,The Vanguard Information Technology ETF (VGT) ...,https://finnhub.io/api/news?id=d1856e9b83f5eed...,2026-03-26 16:54:58


In [ ]:
# Company Peers
print(finnhub_client.company_peers('AAPL'))

In [ ]:
# Company Profile 2
print(finnhub_client.company_profile2(symbol='AAPL'))

In [ ]:
# List country
print(finnhub_client.country())

In [ ]:
# Crypto Exchange
print(finnhub_client.crypto_exchanges())

In [ ]:
# Crypto symbols
print(finnhub_client.crypto_symbols('BINANCE'))

In [ ]:
# Financials as reported
print(finnhub_client.financials_reported(symbol='AAPL', freq='annual'))

In [ ]:
# Forex exchanges
print(finnhub_client.forex_exchanges())

In [ ]:
# Forex symbols
print(finnhub_client.forex_symbols('OANDA'))

In [ ]:
# General news
print(finnhub_client.general_news('forex', min_id=0))

In [ ]:
# IPO calendar
print(finnhub_client.ipo_calendar(_from="2020-05-01", to="2020-06-01"))

In [ ]:
# Quote
print(finnhub_client.quote('AAPL'))

In [ ]:
# Recommendation trends
print(finnhub_client.recommendation_trends('AAPL'))

In [ ]:
# Stock symbols
print(finnhub_client.stock_symbols('US')[0:5])

In [ ]:
# Earnings Calendar
print(finnhub_client.earnings_calendar(_from="2026-03-16", to="2026-03-20", symbol="AAPL", international=False))

In [ ]:
# Covid-19
print(finnhub_client.covid19())

In [ ]:
# FDA Calendar
print(finnhub_client.fda_calendar())

In [ ]:
# Symbol lookup
print(finnhub_client.symbol_lookup('apple'))

In [ ]:
# Visa application
print(finnhub_client.stock_visa_application("AAPL", "2021-01-01", "2022-06-15"))

In [ ]:
# Insider sentiment
print(finnhub_client.stock_insider_sentiment('AAPL', '2021-01-01', '2022-03-01'))

In [ ]:
# USA Spending
print(finnhub_client.stock_usa_spending("LMT", "2021-01-01", "2022-06-15"))

In [ ]:
## Market Holday / Status
print(finnhub_client.market_holiday(exchange='US'))
print(finnhub_client.market_status(exchange='US'))

## Paid Endpoints